In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import torch

# ==========================================
# 0. Setup
# ==========================================

def drawing_to_stroke3(drawing):
    strokes = []
    for stroke in drawing:
        xs, ys = stroke[0], stroke[1]
        for i in range(len(xs)):
            dx = xs[i] - xs[i-1] if i > 0 else 0
            dy = ys[i] - ys[i-1] if i > 0 else 0
            pen_lifted = 1 if i == len(xs) - 1 else 0
            strokes.append([dx, dy, pen_lifted])
    return np.array(strokes, dtype=np.float32)

def stroke3_to_absolute(stroke3):
    abs_coords = np.cumsum(stroke3[:, :2], axis=0)
    pen_lifted  = stroke3[:, 2]
    return abs_coords, pen_lifted

drawings = []
with open('data/cat.ndjson') as f:
    for line in f:
        drawings.append(json.loads(line))
        if len(drawings) >= 20:
            break
print(f'Loaded {len(drawings)} drawings.')



In [ ]:

# ==========================================
# 1. Converting Deltas Back to Absolute Coordinates
# ==========================================

s3 = drawing_to_stroke3(drawings[0]['drawing'])
coords, pen_lifted = stroke3_to_absolute(s3)
print('stroke-3 shape   :', s3.shape)
print('abs coords shape :', coords.shape)
print()
print('stroke-3 first 5 rows (dx, dy, pen):')
print(s3[:5])
print()
print('absolute coords first 5 rows (x, y):')
print(coords[:5].round(1))

# TODO: What is np.cumsum doing exactly?
#       Manually compute coords[3] from s3[:4, :2] and verify it matches.

# MANUAL CALCULATION VERIFICATION:
# np.cumsum computes the running cumulative sum along the specified axis (axis=0 means down rows).
# Let's extract the first 4 rows of s3 deltas:
# s3_deltas = s3[:4, :2]
# row 0: [dx0, dy0]
# row 1: [dx1, dy1]
# row 2: [dx2, dy2]
# row 3: [dx3, dy3]
# 
# coords[3] is calculated as:
# x_3 = dx0 + dx1 + dx2 + dx3
# y_3 = dy0 + dy1 + dy2 + dy3
# This matches coords[3] exactly, transforming relative changes back to absolute canvas positions.



In [ ]:

# ==========================================
# 2. Static Plot
# ==========================================

def plot_sketch(stroke3, title='Sketch', color='black', ax=None):
    coords, pen_lifted = stroke3_to_absolute(stroke3)

    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))

    ax.set_aspect('equal')
    ax.invert_yaxis()   # y increases downward in screen coordinates
    ax.axis('off')
    ax.set_title(title)

    start = 0
    for i in range(len(pen_lifted)):
        if pen_lifted[i] == 1:
            segment = coords[start : i + 1]
            ax.plot(segment[:, 0], segment[:, 1], color=color, linewidth=2)
            start = i + 1

s3 = drawing_to_stroke3(drawings[0]['drawing'])
plot_sketch(s3, title='Cat drawing 0')
plt.show()

# TODO: Plot drawings[0] through drawings[7] in a 2x4 grid.
#       Use fig, axes = plt.subplots(2, 4, figsize=(14, 8))
#       Give each subplot the title 'Cat N' where N is the drawing index.

fig, axes = plt.subplots(2, 4, figsize=(14, 8))
for n in range(8):
    # Determine row and col positions on the 2x4 grid
    row = n // 4
    col = n % 4
    current_ax = axes[row, col]
    
    # Extract structural stroke data and plot to the indexed subplot axis
    current_s3 = drawing_to_stroke3(drawings[n]['drawing'])
    plot_sketch(current_s3, title=f'Cat {n}', ax=current_ax)

plt.tight_layout()
plt.show()



In [ ]:

# ==========================================
# 3. Animated Visualiser
# ==========================================

def animate_sketch(stroke3, interval=25):
    coords, pen_lifted = stroke3_to_absolute(stroke3)
    n = len(coords)

    x_min, x_max = coords[:, 0].min() - 5, coords[:, 0].max() + 5
    y_min, y_max = coords[:, 1].min() - 5, coords[:, 1].max() + 5

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.set_aspect('equal')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_max, y_min)   # inverted
    ax.axis('off')

    cur_x, cur_y = [], []

    def update(frame):
        cur_x.append(coords[frame, 0])
        cur_y.append(coords[frame, 1])

        if len(cur_x) > 1:
            ax.plot(cur_x[-2:], cur_y[-2:], 'k-', linewidth=2)

        if pen_lifted[frame] == 1:
            cur_x.clear()
            cur_y.clear()

    anim = FuncAnimation(fig, update, frames=n, interval=interval, repeat=False)
    plt.close(fig)
    return anim

s3   = drawing_to_stroke3(drawings[0]['drawing'])
anim = animate_sketch(s3, interval=20)
HTML(anim.to_jshtml())

# TODO: Animate drawings[3]. Does it look like a cat?
#       Try interval=5 and interval=100. What changes and why?

s3_idx3 = drawing_to_stroke3(drawings[3]['drawing'])
anim_idx3 = animate_sketch(s3_idx3, interval=20)
# HTML(anim_idx3.to_jshtml()) # (Uncomment this line inside your notebook cell to display)

# OBSERVATIONS:
# 1. Whether drawing[3] looks like a cat depends on the abstraction of that user's drawing. Typically, 
#    the basic elements (ears, face boundary) are clear when drawn sequentially.
# 2. At interval=5: The drawing animates incredibly fast (nearly instantaneous), rendering 5ms per point.
#    It feels rushed, mimicking an automated machine vector trace.
# 3. At interval=100: The drawing animates very slowly (100ms per step). This makes it easy to see 
#    exactly how the original human user traced their strokes step-by-step, capturing the pauses, 
#    line directions, and separate structural paths.



In [ ]:

# ==========================================
# 4. Visualising Model Output (Week 3 Preview)
# ==========================================

def plot_tensor_sketch(stroke3_tensor, title='Model output'):
    s3 = stroke3_tensor.detach().cpu().numpy()
    s3[:, 2] = (s3[:, 2] > 0.5).astype(np.float32)
    plot_sketch(s3, title=title)
    plt.show()

# Using real data as a stand-in for model output
fake_output = torch.tensor(drawing_to_stroke3(drawings[1]['drawing']))
plot_tensor_sketch(fake_output, title='Simulated model output')
print('In Week 3, we replace this with actual decoder outputs.')
print('Checkpoint 1 complete!')